# Blood-Cell Count and Detection

## Setup

### Install

In [ ]:
!pip -q install pycocotools albumentations


### Download


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Shenggan/BCCD_Dataset.git"

bccd_dir = REPO_URL.split("/")[-1].replace(".git", "")
root_path = Path.cwd()
repo_path = root_path / bccd_dir

if not repo_path.exists():
    !git clone {REPO_URL}

In [ ]:
bccd_path = repo_path / "BCCD"
img_path = bccd_path / "JPEGImages"
ann_path = bccd_path / "Annotations"

print("Images:", len(list(img_path.glob("*.jpg"))))
print("Annotations:", len(list(ann_path.glob("*.xml"))))


## Preprocess

### Read

In [ ]:
import pandas as pd

csv_path = repo_path / "test.csv"
df = pd.read_csv(csv_path)
df

In [ ]:
df["cell_type"].value_counts()

In [ ]:
df["filename"].nunique()

In [ ]:
sorted(df["cell_type"].unique())

In [ ]:
df.groupby("cell_type")["filename"].nunique()

### Split


In [ ]:
df["path"] = df["filename"].apply(lambda x: img_path / x)
classes = sorted(df["cell_type"].unique())
class_to_int = {name: i + 1 for i, name in enumerate(classes)}
int_to_class = {v: k for k, v in class_to_int.items()}
df["class"] = df["cell_type"].map(class_to_int)

In [ ]:
df = df[(df["ymin"] < df["ymax"]) & (df["xmin"] < df["xmax"])]

In [ ]:
from sklearn.model_selection import train_test_split

images = df["path"].unique()
images.sort()

development_images, test_images = train_test_split(images, test_size=0.2, random_state=42, shuffle=True)
train_images, valid_images = train_test_split(development_images, test_size=0.25, random_state=42, shuffle=True)

print("Train:", len(train_images))
print("Valid:", len(valid_images))
print("Test:", len(test_images))

train_df = df[df["path"].isin(train_images)]
valid_df = df[df["path"].isin(valid_images)]
test_df = df[df["path"].isin(test_images)]

### Augment

In [ ]:
from albumentations import HorizontalFlip, VerticalFlip, Compose, BboxParams

bbox_params = BboxParams(format="pascal_voc")
train_aug = Compose([HorizontalFlip(), VerticalFlip()], bbox_params=bbox_params)
valid_aug = None


### Load

In [ ]:
from typing import Optional

import numpy as np
import torch
from albumentations.core.transforms_interface import BasicTransform
from PIL import Image
from torch.utils.data import Dataset
from torchvision.io import read_image

class BCCDDataset(Dataset):
    def __init__(
        self, df: pd.DataFrame, augment: Optional[BasicTransform] = None
    ):
        self.df = df
        self.augment = augment
        self.images = self.df["path"].unique()
        self.images.sort()

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(
        self, idx: int
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        image_path = self.images[idx]
        image = np.asarray(Image.open(image_path)) / 255
        height, width, channels = image.shape
        records = self.df[self.df["path"] == image_path]

        boxes = records[["xmin", "ymin", "xmax", "ymax"]].values
        labels = records["class"].values

        if self.augment is not None:
            transformed = self.augment(image=image, bboxes=boxes)
            image = transformed["image"]
            boxes = transformed["bboxes"]

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        return image, target

train_dataset = BCCDDataset(train_df, augment=train_aug)
valid_dataset = BCCDDataset(valid_df, augment=valid_aug)


### Draw

In [ ]:
from typing import Iterable
import matplotlib.pyplot as plt


def xyxy_to_xywh(box: Iterable[float]) -> list[float]:
    xmin, ymin, xmax, ymax = [float(v) for v in box]
    return [xmin, ymin, xmax - xmin, ymax - ymin]


def draw_boxes(
    image_tensor: torch.Tensor,
    boxes: torch.Tensor,
    labels: torch.Tensor,
    scores: Optional[torch.Tensor] = None,
    threshold: float = 0.4,
):
    if scores is not None:
        keep = scores >= threshold
        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]

    image = image_tensor.permute(1, 2, 0).detach().numpy()
    height, width = image.shape[0], image.shape[1]
    figsize = width // 80, height // 80
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)

    for i, box in enumerate(boxes):
        xmin, ymin, box_width, box_height = xyxy_to_xywh(box)
        rect = plt.Rectangle((xmin, ymin), box_width, box_height, fill=False)
        ax.add_patch(rect)
        label = labels[i]
        text = int_to_class[int(label)]
        if scores is not None:
            text += f" {float(scores[i]):.2f}"
        ax.text(xmin, ymin, text)
    plt.axis("off")
    plt.show()

In [ ]:
image, target = valid_dataset[0]
draw_boxes(image, target["boxes"], target["labels"])

In [ ]:
image, target = train_dataset[2]
draw_boxes(image, target["boxes"], target["labels"])

### Batch

In [ ]:
from torch.utils.data import DataLoader


BATCH_SIZE = 16


def collate_fn(batch: tuple[torch.Tensor, dict[str, torch.Tensor]]):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device


In [ ]:
for i in train_loader:
    break

## Model


### Build

In [ ]:
import torchvision.models.detection as detection

model = detection.fasterrcnn_mobilenet_v3_large_320_fpn(weights="COCO_V1")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = detection.faster_rcnn.FastRCNNPredictor(
    in_features, len(class_to_int) + 1
)

In [ ]:
with torch.no_grad():
    print(model([image], [target]))

In [ ]:
model = model.to(device)

### Predict

In [ ]:
@torch.no_grad()
def predict_batch(model, images, device):
    model.eval()
    images = [img.to(device) for img in images]
    outputs = model(images)
    outputs = [{k: v.detach().cpu() for k, v in out.items()} for out in outputs]
    return outputs


In [ ]:
sample_imgs, sample_targets = next(iter(valid_loader))
sample_outputs = predict_batch(model, sample_imgs, device)

for i in range(2):
    draw_boxes(sample_imgs[i], **sample_outputs[i], threshold=0.35)


### Loss

In [ ]:
from tqdm import tqdm

def compute_loss(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
):
    model.train()
    running_loss = {}
    loop = tqdm(data_loader, total=len(data_loader), leave=True)
    for images, targets in loop:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
        loss_dict = model(images, targets)
        if running_loss:
            for k, v in loss_dict.items():
                running_loss[k] += float(v.item())
        else:
            running_loss = {k: float(v.item()) for k, v in loss_dict.items()}
    return {k: v / len(data_loader) for k, v in running_loss.items()}

In [ ]:
compute_loss(model, train_loader, device)

### mAP


In [ ]:
from typing import Any


def build_coco_gt(
    dataset: BCCDDataset
) -> dict[str, dict[str, Any]]:
    coco_images = []
    coco_annotations = []
    annotation_id = 1

    for i, (image, target) in enumerate(dataset):
        coco_images.append({
            "id": i,
            "file_name": dataset.images[i],
            "width": image.shape[0],
            "height": image.shape[1],
        })

        for box, label in zip(target["boxes"], target["labels"]):
            xywh = xyxy_to_xywh(box)
            area = float(max(xywh[2], 0.0) * max(xywh[3], 0.0))
            coco_annotations.append({
                "id": annotation_id,
                "image_id": i,
                "category_id": int(label),
                "bbox": xywh,
                "area": area,
                "iscrowd": 0,
            })
            annotation_id += 1

    coco_categories = [{"id": k, "name": v} for k, v in int_to_class.items()]

    coco_dict = {
        "images": coco_images,
        "annotations": coco_annotations,
        "categories": coco_categories,
        "info": {"description": "BCCD validation split"},
    }
    return coco_dict

In [ ]:
coco_gt_dict = build_coco_gt(valid_dataset)

In [ ]:
@torch.no_grad()
def build_coco_predictions(
    model: torch.nn.Module,
    dataset: torch.utils.data.Dataset,
    device: torch.device,
    threshold: float = 0.05,
) -> list[dict[str, Any]]:
    model.eval()
    predictions = []

    for idx in range(len(dataset)):
        image, target = dataset[idx]
        output = predict_batch(model, [image], device)[0]

        keep = output["scores"] >= threshold
        boxes = output["boxes"][keep]
        labels = output["labels"][keep]
        scores = output["scores"][keep]

        for box, label, score in zip(boxes, labels, scores):
            predictions.append({
                "image_id": idx,
                "category_id": int(label.item()),
                "bbox": xyxy_to_xywh(box.tolist()),
                "score": float(score.item()),
            })

    return predictions

In [ ]:
coco_predictions = build_coco_predictions(model, valid_dataset, device)

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def evaluate_coco_map(
    coco_gt_dict, coco_predictions,
):
    coco_gt = COCO()
    coco_gt.dataset = coco_gt_dict
    coco_gt.createIndex()

    if len(coco_predictions) == 0:
        print("No predictions above threshold; COCO mAP is undefined.")
        return {
            "AP@[0.50:0.95]": 0.0,
            "AP@0.50": 0.0,
            "AP@0.75": 0.0,
            "AR@[0.50:0.95]": 0.0,
        }

    print(coco_predictions[0])

    coco_dt = coco_gt.loadRes(coco_predictions)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

In [ ]:
evaluate_coco_map(coco_gt_dict, coco_predictions)

## Train


### Loop

In [ ]:
def train_one_epoch(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    num_epochs: int,
):
    model.train()
    running_loss = 0.0

    loop = tqdm(data_loader, total=len(data_loader), leave=True)
    batches = 0
    for images, targets in loop:
        batches += 1
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=running_loss / batches)

    return running_loss / batches

### Run

In [ ]:
NUM_EPOCHS = 20
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0)

train_losses = []
for epoch in range(NUM_EPOCHS):
    loss = train_one_epoch(model, optimizer, train_loader, device, NUM_EPOCHS)
    train_losses.append(loss)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training loss")
plt.grid(True)
plt.show()


### Evaluate

In [ ]:
compute_loss(model, train_loader, device)

In [ ]:
sample_imgs, sample_targets = next(iter(valid_loader))
sample_outputs = predict_batch(model, sample_imgs, device)

for i in range(2):
    draw_boxes(sample_imgs[i], **sample_outputs[i], threshold=0.35)


In [ ]:
coco_predictions = build_coco_predictions(model, valid_dataset, device)

In [ ]:
evaluate_coco_map(coco_gt_dict, coco_predictions)